# 🏭 MFG Sentinel
## Predictive Maintenance & Failure Diagnosis Command Center

### Project Overview

**MFG Sentinel** is a predictive-maintenance and machine-failure diagnosis solution built around **Snowflake, Snowflake ML, SQL, and Streamlit**.

The solution is designed to transform raw machine operating conditions into an actionable maintenance signal. It uses machine sensor measurements to estimate the probability of an overall machine failure, independently evaluates five known failure modes, ranks the resulting diagnostic signals, and exposes the resulting intelligence through a Streamlit command center.

The underlying predictive-maintenance workflow is centered on the **AI4I 2020 Predictive Maintenance Dataset**. The dataset is loaded into Snowflake and transformed into an analytical feature layer before model training. The resulting Snowflake ML models are then used directly during inference.

The implementation deliberately focuses on the data and capabilities supported by the available dataset. No synthetic ERP records, maintenance histories, production records, machine identities, or OEE components are fabricated.

---

## 🎯 Objective

The objective of MFG Sentinel is to provide a unified machine-health workflow that can:

1. Accept current machine operating conditions.
2. Transform raw sensor values into model-ready features.
3. Predict the probability of an overall machine failure.
4. Diagnose potential failure mechanisms.
5. Rank the failure-mode signals.
6. Generate a machine-health assessment and recommended response.
7. Surface the results through an operational Streamlit interface.
8. Trigger an automated email alert when the predicted failure risk reaches a critical threshold.

The solution therefore follows the principle:

> **Sensor Data → Machine Learning → Diagnosis → Recommended Action → Alert**

---

## 🏗️ End-to-End Solution Pipeline

```text
                         ┌───────────────────────────────┐
                         │       AI4I 2020 Dataset       │
                         │                               │
                         │ Machine sensor observations   │
                         │ Product information           │
                         │ Failure labels                │
                         └───────────────┬───────────────┘
                                         │
                                         ▼
                         ┌───────────────────────────────┐
                         │      Snowflake Ingestion       │
                         │                               │
                         │ RAW_SENSOR_DATA               │
                         └───────────────┬───────────────┘
                                         │
                                         ▼
                         ┌───────────────────────────────┐
                         │      Feature Engineering       │
                         │                               │
                         │ Temperature Delta             │
                         │ Power Proxy                   │
                         │ Wear × Torque Interaction     │
                         └───────────────┬───────────────┘
                                         │
                                         ▼
                   ┌────────────────────────────────────────────┐
                   │              Training Layer                 │
                   │                                            │
                   │  Curated training views are created for:  │
                   │                                            │
                   │  • Overall Machine Failure                │
                   │  • TWF — Tool Wear Failure                │
                   │  • HDF — Heat Dissipation Failure         │
                   │  • PWF — Power Failure                    │
                   │  • OSF — Overstrain Failure                │
                   │  • RNF — Random Failure                   │
                   └────────────────────┬───────────────────────┘
                                        │
                                        ▼
                   ┌────────────────────────────────────────────┐
                   │            Snowflake ML Layer               │
                   │                                            │
                   │          FAILURE_MODEL                     │
                   │                │                           │
                   │      ┌─────────┼─────────┐                 │
                   │      ▼         ▼         ▼                 │
                   │     TWF       HDF       PWF                │
                   │      │         │         │                 │
                   │      └────┬────┴────┬────┘                 │
                   │           ▼         ▼                      │
                   │          OSF       RNF                     │
                   └────────────────────┬───────────────────────┘
                                        │
                                        ▼
                         ┌───────────────────────────────┐
                         │       Model Evaluation        │
                         │                               │
                         │ • Global evaluation metrics  │
                         │ • Feature importance         │
                         │ • Confusion matrix            │
                         │ • Failure-mode evaluation     │
                         └───────────────┬───────────────┘
                                         │
                                         ▼
                         ┌───────────────────────────────┐
                         │       Real-Time Inference      │
                         │                               │
                         │ User supplies sensor values   │
                         │ through the Streamlit app     │
                         └───────────────┬───────────────┘
                                         │
                                         ▼
                   ┌────────────────────────────────────────────┐
                   │        Six-Model Diagnostic Layer          │
                   │                                            │
                   │ Overall Failure Probability               │
                   │ TWF Probability                           │
                   │ HDF Probability                           │
                   │ PWF Probability                           │
                   │ OSF Probability                           │
                   │ RNF Probability                           │
                   └────────────────────┬───────────────────────┘
                                        │
                                        ▼
                         ┌───────────────────────────────┐
                         │   Diagnostic / RCA Layer      │
                         │                               │
                         │ Failure-mode ranking          │
                         │ Sensor context                │
                         │ Machine-health assessment     │
                         │ Recommended action            │
                         └───────────────┬───────────────┘
                                         │
                                         ▼
                         ┌───────────────────────────────┐
                         │       Alert Classification    │
                         │                               │
                         │ NORMAL → WARNING → CRITICAL   │
                         └───────────────┬───────────────┘
                                         │
                              Critical threshold?
                                         │
                                         ▼
                         ┌───────────────────────────────┐
                         │     Automated Email Alert      │
                         │                               │
                         │ Risk + Diagnosis + Action      │
                         └───────────────────────────────┘

## 1. Exploratory Data Analysis

In [ ]:
%%sql -r dataframe_2
SELECT COUNT(*) AS ROW_COUNT
FROM MFG_SENTINEL.PUBLIC.RAW_SENSOR_DATA;

In [ ]:
%%sql -r dataframe_3
SELECT *
FROM MFG_SENTINEL.PUBLIC.RAW_SENSOR_DATA
LIMIT 10;

## 2. Create Schema and Feature Engineering

In [ ]:
%%sql -r dataframe_4
CREATE SCHEMA IF NOT EXISTS MFG_SENTINEL.ANALYTICS;

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE TABLE MFG_SENTINEL.ANALYTICS.MACHINE_FEATURES AS
SELECT
    UDI,
    PRODUCT_ID,
    PRODUCT_TYPE,
    AIR_TEMPERATURE_K,
    PROCESS_TEMPERATURE_K,
    ROTATIONAL_SPEED_RPM,
    TORQUE_NM,
    TOOL_WEAR_MIN,
    MACHINE_FAILURE,
    TWF,
    HDF,
    PWF,
    OSF,
    RNF,

    PROCESS_TEMPERATURE_K - AIR_TEMPERATURE_K
        AS TEMPERATURE_DELTA,

    ROTATIONAL_SPEED_RPM * TORQUE_NM
        AS POWER_PROXY,

    TORQUE_NM * TOOL_WEAR_MIN
        AS WEAR_TORQUE_INTERACTION

FROM MFG_SENTINEL.PUBLIC.RAW_SENSOR_DATA;

In [ ]:
%%sql -r dataframe_6
SELECT *
FROM MFG_SENTINEL.ANALYTICS.MACHINE_FEATURES
LIMIT 10;

## 3. First Layer Failure Model

In [ ]:
%%sql -r dataframe_7
SHOW SNOWFLAKE.ML.CLASSIFICATION;

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE VIEW MFG_SENTINEL.ANALYTICS.FAILURE_TRAINING_DATA AS
SELECT
    AIR_TEMPERATURE_K,
    PROCESS_TEMPERATURE_K,
    ROTATIONAL_SPEED_RPM,
    TORQUE_NM,
    TOOL_WEAR_MIN,
    TEMPERATURE_DELTA,
    POWER_PROXY,
    WEAR_TORQUE_INTERACTION,
    MACHINE_FAILURE
FROM MFG_SENTINEL.ANALYTICS.MACHINE_FEATURES;

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION
    MFG_SENTINEL.ANALYTICS.FAILURE_MODEL (
    INPUT_DATA => SYSTEM$REFERENCE(
        'VIEW',
        'MFG_SENTINEL.ANALYTICS.FAILURE_TRAINING_DATA'
    ),
    TARGET_COLNAME => 'MACHINE_FAILURE'
);

In [ ]:
%%sql -r dataframe_10
CALL MFG_SENTINEL.ANALYTICS.FAILURE_MODEL!SHOW_GLOBAL_EVALUATION_METRICS();

In [ ]:
%%sql -r dataframe_11
CALL MFG_SENTINEL.ANALYTICS.FAILURE_MODEL!SHOW_FEATURE_IMPORTANCE();

In [ ]:
%%sql -r dataframe_12
CALL MFG_SENTINEL.ANALYTICS.FAILURE_MODEL!SHOW_CONFUSION_MATRIX();

### Overall Machine Failure Model

The `FAILURE_MODEL` is the primary classification model in the MFG Sentinel pipeline. It predicts whether a machine is likely to experience an overall failure (`MACHINE_FAILURE = 0` or `1`) using the sensor measurements and engineered features. The model excludes failure-mode labels from its input features to avoid target leakage.

The evaluation indicates strong overall performance, with a **macro F1-score of 0.907** and **macro AUC of 0.979**. The confusion matrix records **45 correctly identified failures** and **16 missed failures**, while only **4 normal observations were incorrectly classified as failures**. Feature-importance results show **Power Proxy, Rotational Speed, and Tool Wear** among the strongest predictive signals.

**Model Evaluation Note:** The overall failure classifier demonstrates strong discrimination between normal and failure cases, with a macro AUC of **0.979** and macro F1-score of **0.907**. The retained evaluation outputs below provide the detailed metrics, feature contributions, and confusion matrix used to assess model behavior.

## 4. Second Layer Failure Predictioning

### 4a. TWF Model

In [ ]:
%%sql -r dataframe_13
CREATE OR REPLACE VIEW MFG_SENTINEL.ANALYTICS.TWF_TRAINING_DATA AS
SELECT
    AIR_TEMPERATURE_K,
    PROCESS_TEMPERATURE_K,
    ROTATIONAL_SPEED_RPM,
    TORQUE_NM,
    TOOL_WEAR_MIN,
    TEMPERATURE_DELTA,
    POWER_PROXY,
    WEAR_TORQUE_INTERACTION,
    TWF
FROM MFG_SENTINEL.ANALYTICS.MACHINE_FEATURES;

In [ ]:
%%sql -r dataframe_14
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION
    MFG_SENTINEL.ANALYTICS.TWF_MODEL (
    INPUT_DATA => SYSTEM$REFERENCE(
        'VIEW',
        'MFG_SENTINEL.ANALYTICS.TWF_TRAINING_DATA'
    ),
    TARGET_COLNAME => 'TWF'
);

In [ ]:
%%sql -r dataframe_15
CALL MFG_SENTINEL.ANALYTICS.TWF_MODEL!SHOW_GLOBAL_EVALUATION_METRICS();

### 4b. HDF Model

In [ ]:
%%sql -r dataframe_16
CREATE OR REPLACE VIEW MFG_SENTINEL.ANALYTICS.HDF_TRAINING_DATA AS
SELECT
    AIR_TEMPERATURE_K,
    PROCESS_TEMPERATURE_K,
    ROTATIONAL_SPEED_RPM,
    TORQUE_NM,
    TOOL_WEAR_MIN,
    TEMPERATURE_DELTA,
    POWER_PROXY,
    WEAR_TORQUE_INTERACTION,
    HDF
FROM MFG_SENTINEL.ANALYTICS.MACHINE_FEATURES;

In [ ]:
%%sql -r dataframe_17
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION
    MFG_SENTINEL.ANALYTICS.HDF_MODEL (
    INPUT_DATA => SYSTEM$REFERENCE(
        'VIEW',
        'MFG_SENTINEL.ANALYTICS.HDF_TRAINING_DATA'
    ),
    TARGET_COLNAME => 'HDF'
);

In [ ]:
%%sql -r dataframe_18
CALL MFG_SENTINEL.ANALYTICS.HDF_MODEL!SHOW_GLOBAL_EVALUATION_METRICS();

### 4c. PWF Model

In [ ]:
%%sql -r dataframe_19
CREATE OR REPLACE VIEW MFG_SENTINEL.ANALYTICS.PWF_TRAINING_DATA AS
SELECT
    AIR_TEMPERATURE_K,
    PROCESS_TEMPERATURE_K,
    ROTATIONAL_SPEED_RPM,
    TORQUE_NM,
    TOOL_WEAR_MIN,
    TEMPERATURE_DELTA,
    POWER_PROXY,
    WEAR_TORQUE_INTERACTION,
    PWF
FROM MFG_SENTINEL.ANALYTICS.MACHINE_FEATURES;

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION
    MFG_SENTINEL.ANALYTICS.PWF_MODEL (
    INPUT_DATA => SYSTEM$REFERENCE(
        'VIEW',
        'MFG_SENTINEL.ANALYTICS.PWF_TRAINING_DATA'
    ),
    TARGET_COLNAME => 'PWF'
);

In [ ]:
CALL MFG_SENTINEL.ANALYTICS.PWF_MODEL!SHOW_GLOBAL_EVALUATION_METRICS();

### 4d. OSF Model

In [ ]:
CREATE OR REPLACE VIEW MFG_SENTINEL.ANALYTICS.OSF_TRAINING_DATA AS
SELECT
    AIR_TEMPERATURE_K,
    PROCESS_TEMPERATURE_K,
    ROTATIONAL_SPEED_RPM,
    TORQUE_NM,
    TOOL_WEAR_MIN,
    TEMPERATURE_DELTA,
    POWER_PROXY,
    WEAR_TORQUE_INTERACTION,
    OSF
FROM MFG_SENTINEL.ANALYTICS.MACHINE_FEATURES;

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION
    MFG_SENTINEL.ANALYTICS.OSF_MODEL (
    INPUT_DATA => SYSTEM$REFERENCE(
        'VIEW',
        'MFG_SENTINEL.ANALYTICS.OSF_TRAINING_DATA'
    ),
    TARGET_COLNAME => 'OSF'
);

In [ ]:
CALL MFG_SENTINEL.ANALYTICS.OSF_MODEL!SHOW_GLOBAL_EVALUATION_METRICS();

### 4e. RNF Model

In [ ]:
CREATE OR REPLACE VIEW MFG_SENTINEL.ANALYTICS.RNF_TRAINING_DATA AS
SELECT
    AIR_TEMPERATURE_K,
    PROCESS_TEMPERATURE_K,
    ROTATIONAL_SPEED_RPM,
    TORQUE_NM,
    TOOL_WEAR_MIN,
    TEMPERATURE_DELTA,
    POWER_PROXY,
    WEAR_TORQUE_INTERACTION,
    RNF
FROM MFG_SENTINEL.ANALYTICS.MACHINE_FEATURES;

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION
    MFG_SENTINEL.ANALYTICS.RNF_MODEL (
    INPUT_DATA => SYSTEM$REFERENCE(
        'VIEW',
        'MFG_SENTINEL.ANALYTICS.RNF_TRAINING_DATA'
    ),
    TARGET_COLNAME => 'RNF'
);

In [ ]:
CALL MFG_SENTINEL.ANALYTICS.RNF_MODEL!SHOW_GLOBAL_EVALUATION_METRICS();

In [ ]:
SELECT
    RNF,
    COUNT(*) AS COUNT
FROM MFG_SENTINEL.ANALYTICS.MACHINE_FEATURES
GROUP BY RNF
ORDER BY RNF;

## 5. Sanity Checking with Sample Inputs

In [ ]:
WITH INPUT_DATA AS (
    SELECT
        300.0 AS AIR_TEMPERATURE_K,
        310.0 AS PROCESS_TEMPERATURE_K,
        1500.0 AS ROTATIONAL_SPEED_RPM,
        50.0 AS TORQUE_NM,
        180.0 AS TOOL_WEAR_MIN,
        10.0 AS TEMPERATURE_DELTA,
        75000.0 AS POWER_PROXY,
        9000.0 AS WEAR_TORQUE_INTERACTION
)
SELECT
    *,
    MFG_SENTINEL.ANALYTICS.FAILURE_MODEL!PREDICT(
        INPUT_DATA => OBJECT_CONSTRUCT(
            'AIR_TEMPERATURE_K', AIR_TEMPERATURE_K,
            'PROCESS_TEMPERATURE_K', PROCESS_TEMPERATURE_K,
            'ROTATIONAL_SPEED_RPM', ROTATIONAL_SPEED_RPM,
            'TORQUE_NM', TORQUE_NM,
            'TOOL_WEAR_MIN', TOOL_WEAR_MIN,
            'TEMPERATURE_DELTA', TEMPERATURE_DELTA,
            'POWER_PROXY', POWER_PROXY,
            'WEAR_TORQUE_INTERACTION', WEAR_TORQUE_INTERACTION
        )
    ) AS FAILURE_PREDICTION
FROM INPUT_DATA;

In [ ]:
%%sql -r dataframe_1
WITH INPUT_DATA AS (
    SELECT
        300.0 AS AIR_TEMPERATURE_K,
        310.0 AS PROCESS_TEMPERATURE_K,
        1500.0 AS ROTATIONAL_SPEED_RPM,
        50.0 AS TORQUE_NM,
        180.0 AS TOOL_WEAR_MIN,
        10.0 AS TEMPERATURE_DELTA,
        75000.0 AS POWER_PROXY,
        9000.0 AS WEAR_TORQUE_INTERACTION
),

PREDICTIONS AS (
    SELECT

        MFG_SENTINEL.ANALYTICS.FAILURE_MODEL!PREDICT(
            INPUT_DATA => OBJECT_CONSTRUCT(
                'AIR_TEMPERATURE_K', AIR_TEMPERATURE_K,
                'PROCESS_TEMPERATURE_K', PROCESS_TEMPERATURE_K,
                'ROTATIONAL_SPEED_RPM', ROTATIONAL_SPEED_RPM,
                'TORQUE_NM', TORQUE_NM,
                'TOOL_WEAR_MIN', TOOL_WEAR_MIN,
                'TEMPERATURE_DELTA', TEMPERATURE_DELTA,
                'POWER_PROXY', POWER_PROXY,
                'WEAR_TORQUE_INTERACTION', WEAR_TORQUE_INTERACTION
            )
        ) AS FAILURE_PREDICTION,

        MFG_SENTINEL.ANALYTICS.TWF_MODEL!PREDICT(
            INPUT_DATA => OBJECT_CONSTRUCT(
                'AIR_TEMPERATURE_K', AIR_TEMPERATURE_K,
                'PROCESS_TEMPERATURE_K', PROCESS_TEMPERATURE_K,
                'ROTATIONAL_SPEED_RPM', ROTATIONAL_SPEED_RPM,
                'TORQUE_NM', TORQUE_NM,
                'TOOL_WEAR_MIN', TOOL_WEAR_MIN,
                'TEMPERATURE_DELTA', TEMPERATURE_DELTA,
                'POWER_PROXY', POWER_PROXY,
                'WEAR_TORQUE_INTERACTION', WEAR_TORQUE_INTERACTION
            )
        ) AS TWF_PREDICTION,

        MFG_SENTINEL.ANALYTICS.HDF_MODEL!PREDICT(
            INPUT_DATA => OBJECT_CONSTRUCT(
                'AIR_TEMPERATURE_K', AIR_TEMPERATURE_K,
                'PROCESS_TEMPERATURE_K', PROCESS_TEMPERATURE_K,
                'ROTATIONAL_SPEED_RPM', ROTATIONAL_SPEED_RPM,
                'TORQUE_NM', TORQUE_NM,
                'TOOL_WEAR_MIN', TOOL_WEAR_MIN,
                'TEMPERATURE_DELTA', TEMPERATURE_DELTA,
                'POWER_PROXY', POWER_PROXY,
                'WEAR_TORQUE_INTERACTION', WEAR_TORQUE_INTERACTION
            )
        ) AS HDF_PREDICTION,

        MFG_SENTINEL.ANALYTICS.PWF_MODEL!PREDICT(
            INPUT_DATA => OBJECT_CONSTRUCT(
                'AIR_TEMPERATURE_K', AIR_TEMPERATURE_K,
                'PROCESS_TEMPERATURE_K', PROCESS_TEMPERATURE_K,
                'ROTATIONAL_SPEED_RPM', ROTATIONAL_SPEED_RPM,
                'TORQUE_NM', TORQUE_NM,
                'TOOL_WEAR_MIN', TOOL_WEAR_MIN,
                'TEMPERATURE_DELTA', TEMPERATURE_DELTA,
                'POWER_PROXY', POWER_PROXY,
                'WEAR_TORQUE_INTERACTION', WEAR_TORQUE_INTERACTION
            )
        ) AS PWF_PREDICTION,

        MFG_SENTINEL.ANALYTICS.OSF_MODEL!PREDICT(
            INPUT_DATA => OBJECT_CONSTRUCT(
                'AIR_TEMPERATURE_K', AIR_TEMPERATURE_K,
                'PROCESS_TEMPERATURE_K', PROCESS_TEMPERATURE_K,
                'ROTATIONAL_SPEED_RPM', ROTATIONAL_SPEED_RPM,
                'TORQUE_NM', TORQUE_NM,
                'TOOL_WEAR_MIN', TOOL_WEAR_MIN,
                'TEMPERATURE_DELTA', TEMPERATURE_DELTA,
                'POWER_PROXY', POWER_PROXY,
                'WEAR_TORQUE_INTERACTION', WEAR_TORQUE_INTERACTION
            )
        ) AS OSF_PREDICTION,

        MFG_SENTINEL.ANALYTICS.RNF_MODEL!PREDICT(
            INPUT_DATA => OBJECT_CONSTRUCT(
                'AIR_TEMPERATURE_K', AIR_TEMPERATURE_K,
                'PROCESS_TEMPERATURE_K', PROCESS_TEMPERATURE_K,
                'ROTATIONAL_SPEED_RPM', ROTATIONAL_SPEED_RPM,
                'TORQUE_NM', TORQUE_NM,
                'TOOL_WEAR_MIN', TOOL_WEAR_MIN,
                'TEMPERATURE_DELTA', TEMPERATURE_DELTA,
                'POWER_PROXY', POWER_PROXY,
                'WEAR_TORQUE_INTERACTION', WEAR_TORQUE_INTERACTION
            )
        ) AS RNF_PREDICTION

    FROM INPUT_DATA
)

SELECT
    FAILURE_PREDICTION:"class"::STRING
        AS FAILURE_CLASS,

    FAILURE_PREDICTION:"probability":"1"::FLOAT
        AS FAILURE_PROBABILITY,

    TWF_PREDICTION:"probability":"1"::FLOAT
        AS TWF_PROBABILITY,

    HDF_PREDICTION:"probability":"1"::FLOAT
        AS HDF_PROBABILITY,

    PWF_PREDICTION:"probability":"1"::FLOAT
        AS PWF_PROBABILITY,

    OSF_PREDICTION:"probability":"1"::FLOAT
        AS OSF_PROBABILITY,

    RNF_PREDICTION:"probability":"1"::FLOAT
        AS RNF_PROBABILITY

FROM PREDICTIONS;